# Clipping with Boundary

In [5]:
import fiona
import rasterio
from rasterio.mask import mask
from rasterio.crs import CRS
from shapely.geometry import shape, mapping
from shapely.ops import transform  # <-- use this, not pyproj.transform
import pyproj

INPUT_P = "/storage/project/r-rbasu31-0/shared/Metro_Boston/01_Input/DSM.tif"
MASK_GEOJSON = "/storage/project/r-rbasu31-0/shared/Metro_Boston/chelsea_5k.geojson"
CLIPPED_P = "/storage/project/r-rbasu31-0/shared/Chelsea/01_Input/DSM.tif"

def clip_geotiff_with_geojson(tif_path, geojson_path, out_path):
    with rasterio.open(tif_path) as src:
        tif_crs = src.crs

        with fiona.open(geojson_path) as geojson:
            geojson_crs = CRS.from_wkt(geojson.crs_wkt)

            transformer = pyproj.Transformer.from_crs(
                geojson_crs, tif_crs, always_xy=True
            )

            geometries = []
            for feature in geojson:
                geom = shape(feature["geometry"])
                projected = transform(transformer.transform, geom)  # <-- fixed
                geometries.append(mapping(projected))

        clipped, out_transform = mask(src, geometries, crop=True, nodata=-9999)

        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": out_transform,
            "nodata": -9999,
            "compress": "lzw",
        })

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(clipped)

    print(f"Saved: {out_path}")

clip_geotiff_with_geojson(INPUT_P, MASK_GEOJSON, CLIPPED_P)

Saved: /storage/project/r-rbasu31-0/shared/Chelsea/01_Input/DSM.tif


# 9.2. Aligning all inputs
Align every input in the same format

In [1]:
import rasterio
import os
import subprocess # To run gdal commands

#==========================CONFIGURE OF ALIGNING RASTERS=======================
data_dir = "/storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Final ATL Files (Aligned)"

bDSM_path = "/storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Final ATL Files (Aligned)/DSM.tif"
output_dir = '/storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Aligned_2' # Where cleaned/aligned rasters will be saved
os.makedirs(output_dir, exist_ok=True)

# List of other input rasters to process
rasters_to_align = {
    'bDSM': data_dir + '/DSM.tif',
    'cDSM': data_dir + '/cDSM.tif',
    'DEM': data_dir + '/DEM.tif',
    'wall_aspect': data_dir + '/wall_aspect.tif',
    'wall_height': data_dir + '/wall_height.tif',
    'lc': data_dir + '/LULC_5class.tif'
}

# parameters for no data
NO_DATA = -9999

svf_extract_dir = data_dir + '/SVF' # Directory to extract SVF files
svf_aligned_output_base_dir  = output_dir + '/SVF'
os.makedirs(svf_aligned_output_base_dir, exist_ok=True)

#==========================CONFIGURE OF ALIGNING RASTERS=======================

# Get target parameters from bDSM
CUSTOM_CRS = "EPSG:26916"
target_res_x = None
target_res_y = None
target_bounds = None
target_width = None
target_height = None
target_nodata = None # Important for consistency



# Run it

In [2]:
try:
    with rasterio.open(bDSM_path) as src:
        target_crs = rasterio.crs.CRS.from_string(CUSTOM_CRS) if CUSTOM_CRS else src.crs
        target_res_x, target_res_y = src.res
        target_bounds = src.bounds
        target_width = src.width
        target_height = src.height
        target_nodata = NO_DATA # 0 is the desired nodata for all aligned rasters # If bDSM has no nodata and you want no nodata in outputs, set to src.nodata
        print(f"Target Parameters (from {os.path.basename(bDSM_path)}):")
        print(f"  CRS: {target_crs}")
        print(f"  Resolution: {target_res_x}, {target_res_y}")
        print(f"  Bounds: {target_bounds}")
        print(f"  Dimensions (W x H): {target_width} x {target_height}")
        print(f"  NoData Value: {target_nodata}")

except rasterio.errors.RasterioIOError as e:
    print(f"ERROR: Could not open bDSM file at {bDSM_path}. Exiting. {e}")
    exit()

print("\n--- Aligning Rasters ---")

for name, input_path in rasters_to_align.items():
    if not os.path.exists(input_path):
        print(f"WARNING: Input file not found at {input_path}. Skipping alignment for {name}.")
        continue
        
    output_path = os.path.join(output_dir, f'{os.path.basename(input_path).replace(".tif", ".tif")}')

    # if os.path.exists(output_path):
    #     print(f"Skipping alignment for {os.path.basename(input_path)} as aligned output already exists.")
    #     continue 

    print(f"\nProcessing {name} ({os.path.basename(input_path)}) -> {os.path.basename(output_path)}")

    # Construct the gdalwarp command
    gdal_command = [
        'gdalwarp',
        '-overwrite',                  # Overwrite output file if it exists
        '-t_srs', str(target_crs),     # Target CRS
        '-tr', str(target_res_x), str(target_res_y), # Target resolution (pixel size)
        '-te', # Target extent (xmin, ymin, xmax, ymax)
        str(target_bounds.left), str(target_bounds.bottom), str(target_bounds.right), str(target_bounds.top),
        '-ts', str(target_width), str(target_height), # Target dimensions (pixels)
        '-r', 'bilinear',              # Resampling method (e.g., bilinear, cubic, near)
                                       # 'near' for categorical data like LC, 'bilinear' for continuous data
        # Handle NoData: Set output NoData to match bDSM, and tell gdalwarp what input nodata is
        # If input has no nodata, -srcnodata will have no effect.
        # If bDSM has no nodata, then -dstnodata is omitted.
    ]

    # Dynamically add -srcnodata and -dstnodata
    with rasterio.open(input_path) as src_input:
        current_nodata = src_input.nodata
        if current_nodata is not None:
            gdal_command.extend(['-srcnodata', str(current_nodata)])
        if target_nodata is not None:
            if name == 'lc':
                gdal_command.extend(['-dstnodata', '1'])
            else:
                gdal_command.extend(['-dstnodata', str(target_nodata)])
        else:
            # If target bDSM has no nodata, ensure output also has no nodata,
            # or choose a suitable nodata value for the output if needed.
            # For simplicity, if bDSM has no nodata, we won't set -dstnodata,
            # and existing nodata will be reprojected to a value.
            pass

    # Special handling for Land Cover (lc) - use nearest neighbor resampling
    if name == 'lc':
        gdal_command[gdal_command.index('-r') + 1] = 'near' # Change resampling method for LC
        

    gdal_command.extend([input_path, output_path])

    try:
        # Run the gdalwarp command
        print(f"  Running: {' '.join(gdal_command)}")
        result = subprocess.run(gdal_command, check=True, capture_output=True, text=True)
        print("  GDAL Warp Output:\n", result.stdout)
        if result.stderr:
            print("  GDAL Warp Errors (if any):\n", result.stderr)

        # Verify output
        with rasterio.open(output_path) as aligned_src:
            print(f"  Aligned {name} CRS: {aligned_src.crs == target_crs}")
            print(f"  Aligned {name} Shape: {aligned_src.shape == (target_height, target_width)}")
            print(f"  Aligned {name} Res: {aligned_src.res == (target_res_x, target_res_y)}")
            # Check bounds within a tolerance due to floating point precision
            bounds_match = all(abs(getattr(aligned_src.bounds, attr) - getattr(target_bounds, attr)) < 1e-6 for attr in ['left', 'bottom', 'right', 'top'])
            print(f"  Aligned {name} Bounds Match: {bounds_match}")

    except subprocess.CalledProcessError as e:
        print(f"ERROR: GDAL Warp failed for {name}: {e}")
        print("  STDOUT:", e.stdout)
        print("  STDERR:", e.stderr)
    except Exception as e:
        print(f"An unexpected error occurred during processing {name}: {e}")

print("\n--- Aligning SVF Rasters ---")
for root, _, files in os.walk(svf_extract_dir):
            for file in files:
                if file.lower().endswith(('.tif', '.tiff', '.img')): # Add other raster extensions if needed
                    svf_raster_path = os.path.join(root, file)
                    svf_output_file_path = os.path.join(svf_aligned_output_base_dir, f'{os.path.basename(svf_raster_path).replace(".tif", ".tif")}')
                    
                    if os.path.exists(svf_output_file_path):
                        print(f"Skipping alignment for {os.path.basename(svf_raster_path)} as aligned output already exists.")
                        continue 
                    
                    print(f"\nProcessing {name} ({os.path.basename(svf_raster_path)}) -> {os.path.basename(svf_output_file_path)}")
                    svf_gdal_command = [
                        'gdalwarp',
                        '-overwrite',
                        '-t_srs', str(target_crs),
                        '-tr', str(target_res_x), str(target_res_y),
                        '-te',
                        str(target_bounds.left), str(target_bounds.bottom), str(target_bounds.right), str(target_bounds.top),
                        '-ts', str(target_width), str(target_height),
                        '-r', 'bilinear', # SVF values are continuous, so bilinear is suitable
                    ]
                    try:
                        with rasterio.open(svf_raster_path) as src_svf: # Use a new variable for SVF source
                            current_nodata_svf = src_svf.nodata
                            if current_nodata_svf is not None:
                                svf_gdal_command.extend(['-srcnodata', str(current_nodata_svf)])
                            if target_nodata is not None:
                                svf_gdal_command.extend(['-dstnodata', str(target_nodata)])
        
                        svf_gdal_command.extend([svf_raster_path, svf_output_file_path])
        
                        # Run the gdalwarp command for SVF
                        print(f"  Running: {' '.join(svf_gdal_command)}")
                        result = subprocess.run(svf_gdal_command, check=True, capture_output=True, text=True)
                        print("  GDAL Warp Output:\n", result.stdout)
                        if result.stderr:
                            print("  GDAL Warp Errors (if any):\n", result.stderr)
        
                        # Verify output for SVF
                        with rasterio.open(svf_output_file_path) as aligned_src_svf: # Use a new variable for aligned SVF source
                            print(f"  Aligned SVF CRS: {aligned_src_svf.crs == target_crs}")
                            print(f"  Aligned SVF Shape: {aligned_src_svf.shape == (target_height, target_width)}")
                            print(f"  Aligned SVF Res: {aligned_src_svf.res == (target_res_x, target_res_y)}")
                            bounds_match = all(abs(getattr(aligned_src_svf.bounds, attr) - getattr(target_bounds, attr)) < 1e-6 for attr in ['left', 'bottom', 'right', 'top'])
                            print(f"  Aligned SVF Bounds Match: {bounds_match}")
        
                    except subprocess.CalledProcessError as e:
                        print(f"ERROR: GDAL Warp failed for SVF file {os.path.basename(svf_raster_path)}: {e}")
                        print("  STDOUT:", e.stdout)
                        print("  STDERR:", e.stderr)
                    except Exception as e:
                        print(f"An unexpected error occurred during processing SVF file {os.path.basename(svf_raster_path)}: {e}")




print("\n--- Alignment Complete ---")
print(f"Aligned rasters are saved in: {output_dir}")
print(f"Aligned SVF rasters are saved in: {svf_aligned_output_base_dir}")

Target Parameters (from DSM.tif):
  CRS: EPSG:26916
  Resolution: 0.9971159401509484, 0.9971159401509484
  Bounds: BoundingBox(left=738216.033922031, bottom=3735260.9832422086, right=742716.0181599323, top=3740586.579478555)
  Dimensions (W x H): 4513 x 5341
  NoData Value: -9999

--- Aligning Rasters ---

Processing bDSM (DSM.tif) -> DSM.tif
  Running: gdalwarp -overwrite -t_srs EPSG:26916 -tr 0.9971159401509484 0.9971159401509484 -te 738216.033922031 3735260.9832422086 742716.0181599323 3740586.579478555 -ts 4513 5341 -r bilinear -srcnodata -9999.0 -dstnodata -9999 /storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Final ATL Files (Aligned)/DSM.tif /storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Aligned_2/DSM.tif
  GDAL Warp Output:
 Creating output file that is 4513P x 5341L.
Processing /storage/project/r-rbasu31-0/shared/Metro_Boston/Worldcup/Atlanta/Final ATL Files (Aligned)/DSM.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - don